# EM 算法实战：三硬币模型

本 Notebook 是 [一文了解 EM 算法](./一文了解%20EM%20算法.md) 的配套代码实践。

我们将**从零实现**文档中描述的“三硬币模型” EM 算法，不依赖任何高级机器学习库（如 scikit-learn），仅使用 `numpy` 进行数值计算。这将帮助你深入理解 EM 算法内部的“E步”和“M步”是如何交替工作的。

## 1. 问题回顾

假设有三枚硬币 A、B、C，正面概率分别为 $\pi, p, q$。
- 掷硬币 A，若正面（概率 $\pi$）选 B，否则（概率 $1-\pi$）选 C。
- 掷选中的硬币（B 或 C），记录结果（1 为正面，0 为反面）。

我们观测到 10 次结果：`1, 1, 0, 1, 0, 0, 1, 0, 1, 1`。
目标：估计参数 $\theta = (\pi, p, q)$。

In [1]:
import numpy as np

# 观测数据
Y = np.array([1, 1, 0, 1, 0, 0, 1, 0, 1, 1])
n = len(Y)

print(f"观测数据 Y: {Y}")
print(f"样本数量 n: {n}")

观测数据 Y: [1 1 0 1 0 0 1 0 1 1]
样本数量 n: 10


## 2. 初始化参数

为了复现文档中的计算过程，我们使用相同的初始值：
- $\pi_0 = 0.4$
- $p_0 = 0.6$
- $q_0 = 0.7$

In [2]:
# 初始化参数
pi = 0.4
p = 0.6
q = 0.7

theta = (pi, p, q)
print(f"初始参数: pi={pi}, p={p}, q={q}")

初始参数: pi=0.4, p=0.6, q=0.7


## 3. 实现 E 步 (Expectation)

E 步的目标是计算隐变量 $Z$ 的后验概率（即观测数据来自硬币 B 的概率），记为 $\mu$。

公式：
$$
\mu_j = \frac{\pi p^{y_j} (1-p)^{1-y_j}}{\pi p^{y_j} (1-p)^{1-y_j} + (1-\pi) q^{y_j} (1-q)^{1-y_j}}
$$

In [3]:
def e_step(Y, pi, p, q):
    """
    E步：计算每个样本来自硬币B的概率 mu
    """
    mu = []
    for y in Y:
        # 计算分子：P(y, z=B | theta) = P(z=B)*P(y|z=B) = pi * [p^y * (1-p)^(1-y)]
        prob_B = pi * (p**y) * ((1-p)**(1-y))
        
        # 计算分母的一部分：P(y, z=C | theta) = P(z=C)*P(y|z=C) = (1-pi) * [q^y * (1-q)^(1-y)]
        prob_C = (1-pi) * (q**y) * ((1-q)**(1-y))
        
        # 计算后验概率 mu = P(z=B | y, theta)
        mu_val = prob_B / (prob_B + prob_C)
        mu.append(mu_val)
    
    return np.array(mu)

# 测试一下第一轮 E 步的结果
mu_1 = e_step(Y, pi, p, q)
print("第一轮 E 步结果 (mu):", np.round(mu_1, 4))

# 验证文档中的计算细节
# y=1 时，mu 应该约为 0.3636
# y=0 时，mu 应该约为 0.4706
print(f"验证 y=1: {mu_1[0]:.4f}")
print(f"验证 y=0: {mu_1[2]:.4f}")

第一轮 E 步结果 (mu): [0.3636 0.3636 0.4706 0.3636 0.4706 0.4706 0.3636 0.4706 0.3636 0.3636]
验证 y=1: 0.3636
验证 y=0: 0.4706


## 4. 实现 M 步 (Maximization)

M 步的目标是根据计算出的 $\mu$ 来更新参数 $\pi, p, q$。

公式：
$$
\pi_{new} = \frac{1}{n} \sum_{j=1}^{n} \mu_j
$$
$$
p_{new} = \frac{\sum_{j=1}^{n} \mu_j y_j}{\sum_{j=1}^{n} \mu_j}
$$
$$
q_{new} = \frac{\sum_{j=1}^{n} (1-\mu_j) y_j}{\sum_{j=1}^{n} (1-\mu_j)}
$$

In [4]:
def m_step(Y, mu):
    """
    M步：根据 mu 更新参数 pi, p, q
    """
    n = len(Y)
    
    # 更新 pi
    pi_new = np.sum(mu) / n
    
    # 更新 p
    p_new = np.sum(mu * Y) / np.sum(mu)
    
    # 更新 q
    q_new = np.sum((1 - mu) * Y) / np.sum(1 - mu)
    
    return pi_new, p_new, q_new

# 测试一下第一轮 M 步的结果
pi_new, p_new, q_new = m_step(Y, mu_1)
print(f"第一轮 M 步结果: pi={pi_new:.4f}, p={p_new:.4f}, q={q_new:.4f}")

# 验证文档中的数值
# pi 应该约为 0.4064
# p  应该约为 0.5368
# q  应该约为 0.6433

第一轮 M 步结果: pi=0.4064, p=0.5368, q=0.6432


## 5. 完整的 EM 迭代过程

将 E 步和 M 步结合起来，进行多次迭代，直到参数收敛。

In [5]:
# 重新初始化
pi, p, q = 0.4, 0.6, 0.7
max_iter = 10

print(f"{'-'*50}")
print(f"{'Iter':<5} | {'pi':<10} | {'p':<10} | {'q':<10}")
print(f"{'-'*50}")
print(f"{0:<5} | {pi:<10.4f} | {p:<10.4f} | {q:<10.4f}")

for i in range(1, max_iter + 1):
    # E Step
    mu = e_step(Y, pi, p, q)
    
    # M Step
    pi, p, q = m_step(Y, mu)
    
    print(f"{i:<5} | {pi:<10.4f} | {p:<10.4f} | {q:<10.4f}")

print(f"{'-'*50}")
print("迭代完成！")

--------------------------------------------------
Iter  | pi         | p          | q         
--------------------------------------------------
0     | 0.4000     | 0.6000     | 0.7000    
1     | 0.4064     | 0.5368     | 0.6432    
2     | 0.4064     | 0.5368     | 0.6432    
3     | 0.4064     | 0.5368     | 0.6432    
4     | 0.4064     | 0.5368     | 0.6432    
5     | 0.4064     | 0.5368     | 0.6432    
6     | 0.4064     | 0.5368     | 0.6432    
7     | 0.4064     | 0.5368     | 0.6432    
8     | 0.4064     | 0.5368     | 0.6432    
9     | 0.4064     | 0.5368     | 0.6432    
10    | 0.4064     | 0.5368     | 0.6432    
--------------------------------------------------
迭代完成！


## 6. 总结

通过这个简单的 Python 实现，我们可以清晰地看到 EM 算法是如何工作的：

1.  **初始猜测**：我们对参数有了一个初始的（可能是错误的）猜测。
2.  **E 步 (填补数据)**：利用这个猜测，我们估计了每个数据样本来自硬币 B 的可能性 ($\mu$)。
3.  **M 步 (优化参数)**：假设这些可能性就是真实的权重，我们重新计算了硬币 A、B、C 的概率。
4.  **收敛**：随着迭代进行，参数逐渐稳定。

虽然这个例子很简单，但它包含了 EM 算法的所有核心要素。在更复杂的场景（如 GMM 高斯混合模型）中，原理完全相同，只是概率密度公式和参数更新公式变得更复杂而已。